In [ ]:
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
zip_name = "power_data.zip"

if not os.path.exists('household_power_consumption.txt'):
    urllib.request.urlretrieve(url, zip_name)
    
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall() 
        
    os.remove(zip_name)
    print("готово")
else:
    print("файл 'household_power_consumption.txt' вже знаходиться у правильній папці")

Файл 'household_power_consumption.txt' вже знаходиться у правильній папці!


Здійснити data cleaning

In [2]:
import pandas as pd
import numpy as np
import timeit

filepath = 'household_power_consumption.txt'

def load_clean(filepath):

    df = pd.read_csv(filepath, sep = ';', na_values = ['?'], low_memory = False)

    df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format = '%d/%m/%Y %H:%M:%S')
    df.drop(columns = ['Date', 'Time'], inplace = True)

    df.dropna(inplace = True)

    return df

In [3]:
df = load_clean(filepath)
display(df.head())

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
0,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.

In [4]:
def sample_1(df):
    start_time = timeit.default_timer()
    df_1 = df[df['Global_active_power'] > 5]
    
    end_time = timeit.default_timer()
    time = (end_time) - (start_time)
    return df_1, time

In [5]:
result_1, time_1 = sample_1(df)
print(f"time: {time_1} sec")
display(result_1.head())

time: 0.006670799921266735 sec


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
11,5.412,0.470,232.78,23.2,0.0,1.0,17.0,2006-12-16 17:35:00
12,5.224,0.478,232.99,22.4,0.0,1.0,16.0,2006-12-16 17:36:00


Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.

In [6]:
def sample_2(df):
    start_time = timeit.default_timer()
    df_2 = df[(df['Global_intensity'].between(19, 20))&(df['Sub_metering_2'] > df['Sub_metering_3'])]

    end_time = timeit.default_timer()
    time = end_time - start_time

    return df_2, time

In [7]:
result_2, time_2 = sample_2(df)
print(f"time: {time_2} sec")
display(result_2.head())

time: 0.018700100015848875 sec


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
45,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00
460,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00
464,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00
475,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00
476,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00


Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії

In [12]:
def sample_3(df):
    start_time = timeit.default_timer()
    df_3 = df.sample(n = 500000, replace = False)

    df_stats = df_3[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    end_time = timeit.default_timer()
    time = end_time - start_time
    return df_stats, time

In [17]:
result_3, time_3 = sample_3(df)
print(f'time: {time_3} sec')
display(result_3)

time: 0.44903869996778667 sec


Sub_metering_1    1.121232
Sub_metering_2    1.292234
Sub_metering_3    6.459146
dtype: float64

Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [19]:
def sample_4(df):
    start_time = timeit.default_timer()

    df_fltrd = df[(df['datetime'].dt.hour >= 18)&(df['Global_active_power'] > 6)&(df['Sub_metering_2'] > df['Sub_metering_1'])&(df['Sub_metering_2'] > df['Sub_metering_3'])]

    mid = len(df_fltrd) // 2
    half_1 = df_fltrd.iloc[:mid:3]
    half_2 = df_fltrd.iloc[mid::4]

    fin_res = pd.concat([half_1, half_2])
    end_time = timeit.default_timer()
    time = end_time - start_time
    return fin_res, time

In [21]:
result_4, time_4 = sample_4(df)
print(f'time : {time_4} sec')
display(result_4.head())

time : 0.09548030002042651 sec


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
41,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00
44,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00
17494,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00
17498,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00
17501,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00


встановлюю потрібний пакет

In [23]:
%pip install scikit-learn

     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ------ --------------------------------- 10.2/61.0 kB ? eta -:--:--
     ------ --------------------------------- 10.2/61.0 kB ? eta -:--:--
     ------------------------- ------------ 41.0/61.0 kB 326.8 kB/s eta 0:00:01
     ------------------------- ------------ 41.0/61.0 kB 326.8 kB/s eta 0:00:01
     -------------------------------------- 61.0/61.0 kB 249.8 kB/s eta 0:00:00
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.1/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.1/8.0 MB ? eta -:--:--
    --------------------------------------- 0.2/8.0 MB 2.0 MB/s eta 0:00:04
    --------------------------------------- 0.2/8.0 MB 2.0 MB/s eta 0:00:04
   -- ------------------------------------- 0.5/8.0 MB 2.2 MB/s eta 0:00:04
   -- ------------------------------------- 0.5/8.0 MB 2.2 MB/s eta 0:00:04
   ---- -----------------------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

створюю копії фрейму

In [25]:
df_std = result_2.copy() 
df_norm = result_2.copy()

Пронормувати та стандартизувати вибраний датасет

1.стандартизація

In [27]:
numeric_cols = [
    'Global_active_power', 'Global_reactive_power', 'Voltage', 
    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
]

std_scaler = StandardScaler()
df_std[numeric_cols] = std_scaler.fit_transform(df_std[numeric_cols])

display(df_std[numeric_cols].head(3))

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,-1.120974,-0.483627,-0.690133,-1.416574,-0.435351,0.079434,0.544519
460,-0.048414,0.217533,0.476645,0.358152,-0.435351,-1.443879,-1.343472
464,0.278807,-0.667537,0.998625,0.358152,-0.435351,-0.555280,-1.343472


2.нормування

In [28]:
minmax_scaler = MinMaxScaler()
df_norm[numeric_cols] = minmax_scaler.fit_transform(df_norm[numeric_cols])

display(df_norm[numeric_cols].head(3))

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,0.539648,0.128302,0.357955,0.0,0.0,0.480000,0.533333
460,0.669604,0.243396,0.519886,0.6,0.0,0.160000,0.000000
464,0.709251,0.098113,0.592330,0.6,0.0,0.346667,0.000000


Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів

In [ ]:
col_1 = 'Global_active_power'
col_2 = 'Global_intensity'

pearson_corr = result_2[col_1].corr(result_2[col_2], method='pearson')

spearman_corr = result_2[col_1].corr(result_2[col_2], method='spearman')

print(f"кореляція Пірсона: {pearson_corr:.4f}")
print(f"кореляція Спірмена: {spearman_corr:.4f}")

Кореляція Пірсона: 0.6995
Кореляція Спірмена: 0.7285


Провести One Hot Encoding категоріального атрибута

відділяю день тижня з колонки datetime і проводжу для цього атрибута One Hot Encoding

In [31]:
result_2['day_of_week'] = result_2['datetime'].dt.day_name()

result_2_ohe = pd.get_dummies(result_2, columns=['day_of_week'])
display(result_2_ohe.head())

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime,day_of_week_Friday,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday
45,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00,False,False,True,False,False,False,False
460,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00,False,False,False,True,False,False,False
464,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00,False,False,False,True,False,False,False
475,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00,False,False,False,True,False,False,False
476,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00,False,False,False,True,False,False,False
